# Latency-Aware Framework - Cloud Training Notebook

Use this notebook to train your models on **Google Colab** or **Kaggle** using a free GPU.

## 1. Setup Environment
If using Google Colab, mount your Google Drive.

In [ ]:
# IF ON COLAB, RUN THIS CELL:
from google.colab import drive
drive.mount('/content/drive')

# Change this path to where you uploaded the 'Research' folder
%cd /content/drive/MyDrive/Research

## 2. Unzip Data (Recommended for Speed)
Reading thousands of images from Drive is slow. It's better to unzip to the local VM (`/content/`).
Ensure you have uploaded `dataset.zip` inside your Research folder on Drive.

In [ ]:
# Unzip dataset to local VM
!unzip -q -n dataset.zip -d /content/local_data

# Check extraction structure
!ls /content/local_data

## 3. Verify GPU
Make sure you are connected to a GPU instance.

In [ ]:
!nvidia-smi

## 4. Install Dependencies

In [ ]:
!pip install thop opencv-python

## 5. Train Models
We will train 3 models: Baseline U-Net, DSC-UNet, and MobileUNet.

**Note on Paths**: We use `--data-dir /content/local_data/dataset` assuming your zip contained the `dataset` folder. If `ls` above shows `syntax` and `stenosis` directly in `local_data`, remove `/dataset` from the path.

### 5.1. Baseline U-Net

In [ ]:
# Train Baseline U-Net
!python src/train.py --epochs 50 --batch-size 8 --learning-rate 1e-4 --scale 0.5 --data-dir /content/local_data/dataset

### 5.2. MobileUNet (Best Candidate)

In [ ]:
# Train MobileUNet
!python src/train.py --model mobileunet --epochs 50 --batch-size 16 --learning-rate 1e-4 --scale 0.5 --data-dir /content/local_data/dataset

### 5.3. DSC-UNet (Alternative Lightweight)

In [ ]:
!python src/train.py --model dscunet --epochs 50 --batch-size 8 --learning-rate 1e-4 --scale 0.5 --data-dir /content/local_data/dataset

## 6. Benchmark on GPU
Check how fast these models run on a Tesla T4 GPU.

In [ ]:
!python src/benchmark.py --device cuda

## 7. Phase 8: Knowledge Distillation (The "Super-Model" Strategy)
Here we aim for **max accuracy** while keeping **max speed**.

1.  **Train Teacher**: `DeepLabV3+ with ResNet-101`. (Huge, Slow, Accurate).
2.  **Train Student**: `MobileUNet-v3`. (Small, Fast). Use the Teacher to guide it.

In [ ]:
# Step 1: Train the Teacher (Takes ~1-2 hours on Colab)
# We use a small batch size because ResNet101 is memory hungry.
# CRITICAL: We pass --data-dir because on Colab paths are different.
!python src/train_teacher.py --epochs 30 --batch-size 4 --data-dir /content/local_data/dataset

In [ ]:
# Step 2: Knowledge Distillation (Train Student using Teacher)
# This will create 'checkpoints/student_distilled/student_epoch50.pth'
!python src/train_distillation.py --teacher checkpoints/teacher/teacher_best.pth --epochs 50 --batch-size 8 --data-dir /content/local_data/dataset